# Week 26 workbook: causal reasoning from noisy biological environments

Read [the teaching guide](teaching_guide.md) first. Sources and provenance are in
[the research dossier](../research/causal_biology.md).

The default execution path uses **reference solutions** in `project/solutions.py`.
It is a reproducible reference run, not a graded learner submission. Set
`REFERENCE_MODE=False` only after filling the eight `learner_*` functions below.
Their initial `NotImplementedError` is intentional and will stop an unfinished attempt.
Checks call whichever function you select on meaningful fixtures with explicit expected values.
All data are synthetic; no clinical conclusion or proprietary method is implemented.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from common.plotting import use_rl_style, RL_PALETTE
from common.seed import seed_everything
use_rl_style(); seed_everything(0)
import numpy as np
from project import solutions as ref
REFERENCE_MODE = True


## Exercise 1 ★: distinguish observed association from an effect

**Prompt.** In the 200-unit fixture, mild patients have untreated outcome 2 and
severe patients outcome 0. Treatment adds 1 for every patient. Mild patients
are treated with frequency 20%, severe patients with frequency 80%. Implement
`learner_naive_ate(treatment, outcome)` as the treated mean minus untreated mean.
Predict its sign before running. State why its causal interpretation requires more than
observing both treatment groups.

<details><summary>Hint</summary>Compute two group means, not the average of stratum effects.
A value of -0.2 is possible despite a +1 effect in both strata.</details>

In [ ]:
def learner_naive_ate(treatment, outcome):
    # YOUR CODE HERE; selecting learner mode intentionally stops until implemented.
    raise NotImplementedError

naive_ate = ref.exercise_1_naive_ate if REFERENCE_MODE else learner_naive_ate
c_fixture = np.repeat([0, 1], 100)
t_fixture = np.r_[np.zeros(80), np.ones(20), np.zeros(20), np.ones(80)]
y_fixture = np.where(c_fixture == 0, 2.0, 0.0) + t_fixture
print("Reference mode:", REFERENCE_MODE, "observed difference:", naive_ate(t_fixture, y_fixture))

In [ ]:
assert np.isclose(naive_ate(t_fixture, y_fixture), -0.2)
assert np.isclose(naive_ate([0, 1, 0, 1], [0, 1, 2, 3]), 1.0)


## Exercise 2 ★★: derive and calculate standardization

**Prompt.** Implement `learner_adjusted_ate(treatment, outcome, confounder)` using
sample stratum proportions and within-stratum mean differences. Require both
treatment levels in every stratum; raise `ValueError` if empirical overlap is missing.
List consistency, exchangeability, and positivity separately. Explain why a
nonempty cell does not prove exchangeability or population positivity.

<details><summary>Hint</summary>Sum `P(C=c) * (mean(Y[A=1,C=c])-mean(Y[A=0,C=c]))`.
Use the same target weights for both potential-outcome means.</details>

In [ ]:
def learner_adjusted_ate(treatment, outcome, confounder):
    # YOUR CODE HERE
    raise NotImplementedError

adjusted_ate = ref.exercise_2_adjusted_ate if REFERENCE_MODE else learner_adjusted_ate
print("Standardized effect:", adjusted_ate(t_fixture, y_fixture, c_fixture))

In [ ]:
assert np.isclose(adjusted_ate(t_fixture, y_fixture, c_fixture), 1.0)
try:
    adjusted_ate(c_fixture, y_fixture, c_fixture)
except ValueError:
    pass
else:
    raise AssertionError("An unsupported stratum must stop the calculation")


## Exercise 3 ★★: observational equivalence is an identification limit

**Prompt.** Implement `learner_equivalent_covariances(rho)` returning two 2×2
covariance matrices for `X=U, Y=rho*X+sqrt(1-rho²)*V` and its reversed SCM,
with independent standard-normal disturbances. Validate `abs(rho)<1`.
For `rho=0.8`, derive both expectations under `do(X=1)` without using regression
as a causal oracle. Explain why no additional observational sample distinguishes these two
Gaussian models. Name one possible experimental design and one necessary biological assumption.

<details><summary>Hint</summary>Each variance is 1 and covariance is rho.
In the reverse SCM, intervening on X leaves the assignment `Y=U` untouched.</details>

In [ ]:
def learner_equivalent_covariances(rho=0.8):
    # YOUR CODE HERE
    raise NotImplementedError

equivalent_covariances = (ref.exercise_3_equivalent_covariances if REFERENCE_MODE
                           else learner_equivalent_covariances)
forward, reverse = equivalent_covariances(0.8)
print(forward, reverse, sep="\n")
print("Derived intervention means at do(X=1): forward=0.8, reverse=0.0")

In [ ]:
f, r = equivalent_covariances(0.6)
np.testing.assert_allclose(f, [[1, 0.6], [0.6, 1]])
np.testing.assert_allclose(r, [[1, 0.6], [0.6, 1]])
assert np.isclose(0.8**2 + 0.6**2, 1.0)
# A course-created negative-control smoother can create correlation 1.
assert np.isclose(0.5 / np.sqrt(0.5*0.5), 1.0)


## Exercise 4 ★★: split patients before touching their cells

**Prompt.** Implement `learner_patient_split(patient_ids, environments, ...)`.
Seal environment 2; stratify other environments by patient into train/validation.
Every repeated row for one patient must stay together. Require integer IDs that
are globally unique and reject a patient spanning environments so that the study
design must resolve that case explicitly. In a real multisite study, what other
identifiers could reveal repeated donors?

<details><summary>Hint</summary>Shuffle unique patient IDs inside each training environment.
Return index arrays. A random row split puts related measurement rows on both sides.</details>

In [ ]:
def learner_patient_split(patient_ids, environments, test_environment=2,
                          validation_fraction=0.3, seed=0):
    # YOUR CODE HERE
    raise NotImplementedError

patient_split = ref.exercise_4_patient_split if REFERENCE_MODE else learner_patient_split
patient_ids = np.repeat(np.arange(18), 3)
environments = np.repeat(np.repeat(np.arange(3), 6), 3)
partitions = patient_split(patient_ids, environments, seed=0)
print({name: len(np.unique(patient_ids[idx])) for name, idx in partitions.items()})

In [ ]:
sets = {name: set(patient_ids[idx]) for name, idx in partitions.items()}
assert not (sets['train'] & sets['validation'] or sets['train'] & sets['test']
            or sets['validation'] & sets['test'])
assert sorted(np.concatenate(list(partitions.values()))) == list(range(len(patient_ids)))
assert set(environments[partitions['test']]) == {2}
for idx in partitions.values():
    assert all(np.sum(patient_ids[idx] == pid) == 3 for pid in np.unique(patient_ids[idx]))


## Exercise 5 ★★: missingness and measurement are separate mechanisms

**Prompt.** Implement `learner_impute_fit(features)` returning mean-imputed
training data and observed-only column means. Preserve literal zeros; use `NaN`
as the fixture's explicit missingness marker. Give an all-missing column the
stated teaching fallback 0. Explain why that fallback represents no measured
information. Transform a held-out matrix with the frozen means, without refitting.
Describe a missing-not-at-random mechanism for which this calculation is inadequate.

<details><summary>Hint</summary>Count observed values per column. Use those counts as denominators,
and separately handle count zero. Neither test features nor labels fit these means.</details>

In [ ]:
def learner_impute_fit(features):
    # YOUR CODE HERE
    raise NotImplementedError

impute_fit = ref.exercise_5_impute_fit if REFERENCE_MODE else learner_impute_fit
train_x = np.array([[0, np.nan, np.nan], [2, 4, np.nan]], dtype=float)
filled, means = impute_fit(train_x)
heldout_x = np.array([[100, np.nan, np.nan]], dtype=float)
heldout_filled = np.where(np.isnan(heldout_x), means, heldout_x)
print("Train means", means, "frozen held-out transform", heldout_filled)

In [ ]:
filled_check, means_check = impute_fit(train_x)
np.testing.assert_allclose(means_check, [1, 4, 0])
np.testing.assert_allclose(filled_check, [[0, 4, 0], [2, 4, 0]])
np.testing.assert_allclose(heldout_filled, [[100, 4, 0]])
assert np.isnan(train_x[0, 1])


## Exercise 6 ★★: descriptive invariance diagnostics

**Prompt.** Implement `learner_residual_spread(residuals, environments)` returning
`mean_spread` and `variance_spread`: maximum minus minimum environment mean and
population-divisor variance. Compare residuals `(-1,1)` and `(-2,2)` across two
environments. Explain why an equal mean alone misses distributional change.
Explain why neither spread is a valid ICP p-value, proof of invariance, or graph identifier.

<details><summary>Hint</summary>The two means are 0; variances are 1 and 4.
A full conditional-mechanism claim is much stronger than two residual summaries.</details>

In [ ]:
def learner_residual_spread(residuals, environments):
    # YOUR CODE HERE
    raise NotImplementedError

residual_spread = ref.exercise_6_residual_spread if REFERENCE_MODE else learner_residual_spread
print(residual_spread([-1, 1, -2, 2], [0, 0, 1, 1]))

In [ ]:
assert residual_spread([-1, 1, -2, 2], [0, 0, 1, 1]) == {
    'mean_spread': 0.0, 'variance_spread': 3.0}
assert residual_spread([1, 2, 3], [0, 0, 0]) == {
    'mean_spread': 0.0, 'variance_spread': 0.0}


## Exercise 7 ★★: task-aware training transformation with an ablation

**Prompt.** Implement `learner_task_randomize(features, column=2, probability=1, seed=0)`
by permuting training nuisance values and replacing a seeded random fraction of rows.
Keep other columns and the source matrix unchanged. For probability 1 the nuisance
column's marginal multiset is preserved exactly, while its target relationship changes.
This fixture specifies nuisance identity using knowledge of its SCM. Would applying the
same operation before gene-graph discovery preserve the evidence being analyzed?

<details><summary>Hint</summary>Use a local NumPy generator; make a copy. Never include validation
or test rows in the permutation. Do not call the generated rows independent patients.</details>

In [ ]:
def learner_task_randomize(features, column=2, probability=1.0, seed=0):
    # YOUR CODE HERE
    raise NotImplementedError

task_randomize = ref.exercise_7_task_randomize if REFERENCE_MODE else learner_task_randomize
x_demo = np.arange(600, dtype=float).reshape(200, 3)
x_aug = task_randomize(x_demo, probability=1.0, seed=5)
print("Original/corrupted association:", np.corrcoef(x_demo[:, 0], x_aug[:, 2])[0, 1])

In [ ]:
np.testing.assert_array_equal(task_randomize(x_demo, probability=0), x_demo)
np.testing.assert_array_equal(x_aug[:, :2], x_demo[:, :2])
np.testing.assert_array_equal(np.sort(x_aug[:, 2]), np.sort(x_demo[:, 2]))
assert not np.array_equal(x_aug[:, 2], x_demo[:, 2])
np.testing.assert_array_equal(task_randomize(x_demo, seed=5), x_aug)


## Exercise 8 ★★★: matching objective and transfer under a changed assumption

**Prompt.** A synthetic lead router can contact two of three leads. Their association-based
scores are `(0.9,0.8,0.1)`, while the simulator's contact-versus-no-contact improvements
are `(-0.2,0.5,1.0)`. Implement `learner_matching_priority_value(scores,
potential_improvements,budget=2)` returning selected indices and mean improvement.
Compare the association ranking with ranking by simulator effects. Use deterministic
stable sorting for ties. Why are these simulator effects unavailable in an ordinary
observational log? Explain why the exercise is not a production routing recommendation.

**Open extension.** Run seeds 0–4, then change `target_shift` to 1.0. Freeze
candidate rates and selection criterion before looking at environment 2. Report the raw,
selected-enhancement, no-proxy, and training-mean baselines. Write a 200-word memo on why
improved prediction does not validate a biological causal graph. Identify one negative
control and one relevant perturbation test; do not tune the rule on the test outcome.

<details><summary>Hint</summary>Association selects leads 0 and 1: mean improvement 0.15.
Effect scores select 2 and 1: mean 0.75. A changed target mechanism defeats a blanket
invariance claim even when a nuisance is correctly removed.</details>

In [ ]:
def learner_matching_priority_value(scores, potential_improvements, budget=2):
    # YOUR CODE HERE
    raise NotImplementedError

matching_priority_value = (ref.exercise_8_matching_priority_value if REFERENCE_MODE
                           else learner_matching_priority_value)
scores = np.array([0.9, 0.8, 0.1])
gains = np.array([-0.2, 0.5, 1.0])
association_router = matching_priority_value(scores, gains, 2)
effect_router = matching_priority_value(gains, gains, 2)
print(association_router, effect_router)

In [ ]:
np.testing.assert_array_equal(association_router['selected'], [0, 1])
assert np.isclose(association_router['value'], 0.15)
np.testing.assert_array_equal(effect_router['selected'], [2, 1])
assert np.isclose(effect_router['value'], 0.75)
assert matching_priority_value([1, 1], [2, -2], 1)['value'] == 2


## Run the bounded mini-project

The ATE calculation is a **separate confounded fixture** from the prediction experiment.
The prediction model uses known synthetic nuisance identity; no PC, FCI, S-FCI or ICP
algorithm is implemented. Imputation and scaling are training-only; candidate selection
uses worst-environment validation MSE; environment 2 is read after selection.
The no-proxy ablation can beat the validation-selected model on the new environment.
That finding is useful evidence about the limit of source validation, not an error to hide.


In [ ]:
from project.train import train
result = train(seed=0)
for key in ['true_ate', 'naive_ate', 'adjusted_ate',
            'selected_randomization_probability', 'baseline_test_mse',
            'enhanced_test_mse', 'stable_features_ablation_test_mse',
            'enhanced_test_mse_patient_bootstrap_95pct']:
    print(key, result[key])
assert abs(result['adjusted_ate']-1) < 0.05
assert result['enhanced_test_mse'] < 0.8*result['baseline_test_mse']


In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(result['baseline_training_loss'], label='Raw ridge', color=RL_PALETTE['blue'])
axes[0].plot(result['enhanced_training_loss'], label='Selected randomization', color=RL_PALETTE['teal'])
axes[0].set(xlabel='Gradient step', ylabel='Training objective')
axes[0].legend()
axes[1].bar(['Raw', 'Enhanced', 'No proxy'],
            [result[k] for k in ['baseline_test_mse','enhanced_test_mse',
                                 'stable_features_ablation_test_mse']],
            color=[RL_PALETTE[k] for k in ['blue','teal','gold']])
axes[1].set(ylabel='Patient-weighted held-out MSE')
fig.tight_layout()
Path('figs').mkdir(exist_ok=True)
fig.savefig('figs/workbook_comparison.pdf', metadata={'CreationDate':None})
plt.show()


## Memo reference outline (interpretation, not an automated grade)

State the effect/prediction question, units, population and intervention. List which
confounders and measurement mechanisms are assumed observed, and what changes across
environments. Give the patient split and which data fitted preprocessing, enhancement and
selection. Show baseline and ablation MSE alongside graph-identification limits. Distinguish
seed-to-seed variation from a patient sampling interval and both from assumption uncertainty.
Name a negative control that detects manufactured marker dependencies, and a targeted,
controlled perturbation in an appropriate cell state. Do not infer drug efficacy from
continuous-marker prediction error. In learner mode, all eight functions must be filled;
passing fixture checks alone is not evidence that the memo's causal assumptions hold.
